In [5]:
import json
import sys
from typing import List, Dict, Any
import numpy as np

In [13]:
def load_task_file(filepath: str) -> dict:
    """Load and parse a task JSON file."""
    try:
        tasks = []
        if filepath.endswith(".json"):
            with open(filepath, "r") as f:
                tasks = json.load(f)
            return tasks
        elif filepath.endswith(".jsonl"):
            with open(filepath, "r") as f:
                for line in f:
                    tasks.append(json.loads(line.strip()))
            return tasks
    except FileNotFoundError:
        print(f"Error: Task file not found: {filepath}")
        sys.exit(1)
    except json.JSONDecodeError as e:
        print(f"Error: Invalid JSON in task file: {e}")
        sys.exit(1)



In [23]:
task_file = "example_task_fill.json"
tasks = [load_task_file(task_file)]
print(f"Loaded {len(tasks)} tasks from {task_file}")

Loaded 1 tasks from example_task_fill.json


In [25]:
tasks[0].get("train", [])

[{'input': [[0, 0, 0], [0, 5, 0], [0, 0, 0]],
  'output': [[1, 1, 1], [1, 5, 1], [1, 1, 1]]},
 {'input': [[0, 0], [0, 7]], 'output': [[1, 1], [1, 7]]}]

In [29]:
for i, task in enumerate(tasks):
    train_pairs = task.get("train", [])
    test_cases = task.get("test", [])
    
    print(f"Task ID: {task.get('id', 'N/A')}")
    print(f"Number of training pairs: {len(train_pairs)}")
    print(f"Number of test cases: {len(test_cases)}")

Task ID: N/A
Number of training pairs: 2
Number of test cases: 1


In [24]:
def _format_examples_for_prompt(task_dict: list) -> str:
        """Format training examples for the prompt."""
        formatted = []
        for idx, example in enumerate(task_dict):
            formatted.append(
                f"**Example {idx + 1}:**\n"
                f"Input shape: {np.array(example['input']).shape}\n"
                f"Input:\n{np.array(example['input'])}\n"
                f"Output shape: {np.array(example['output']).shape}\n"
                f"Output:\n{np.array(example['output'])}\n"
            )
        return "\n".join(formatted)

# examples_str = _format_examples_for_prompt(train_tasks)

In [10]:
tasks = load_task_file('new_dataset.jsonl')

In [19]:
current_task = None
for task in tasks[:1]:
    current_task = task
    

train_pairs = current_task.get("train", [])
test_cases = current_task.get("test", [])
    
print(f"Number of training examples: {len(train_pairs)}, Number of test cases: {len(test_cases)}")

Number of training examples: 3, Number of test cases: 1


In [25]:
examples_str = _format_examples_for_prompt(train_pairs)
test_input_str = str(np.array(test_cases[0]["input"]))

In [38]:

from typing import Optional
from config import config
from openai import AsyncOpenAI
llm_client = AsyncOpenAI(
    api_key=config.LOCAL_LLM_API_KEY,
    base_url=config.LOCAL_LLM_BASE_URL
)
model = config.LOCAL_LLM_MODEL

async def _call_llm(
        system_prompt: str, user_message: str
    ) -> Optional[str]:
        """
        Call the LLM with the given prompts.

        Args:
            system_prompt: System role and instructions.
            user_message: The user's message/query.

        Returns:
            The LLM's response, or None if using mock mode.
        """
        print(f'USE LOCAL_LLM: {config.USE_LOCAL_LLM}, MODEL: {config.LOCAL_LLM_MODEL}')
        # local LLM via HTTP
        if config.USE_LOCAL_LLM:
            try:
                            
                response = await llm_client.chat.completions.create(
                    model=model,
                    messages=[
                        {"role": "system", "content": system_prompt},
                        {"role": "user", "content": user_message},
                    ],
                    temperature=config.TEMPERATURE,
                    # max_tokens=self.router_max_tokens,
                )
                content = (response.choices[0].message.content or "").strip()
                return content
            except Exception as e:
                print(f"Local LLM error: {e}")
                return None

        print(f"Calling LLM with system prompt: 2nd time maybe...")
        
        try:
            response = llm_client.messages.create(
                model=config.MODEL_ID,
                max_tokens=4096,
                temperature=config.TEMPERATURE,
                system=system_prompt,
                messages=[{"role": "user", "content": user_message}],
            )
            return response.content[0].text
        except Exception as e:
            print(f"LLM Error: {e}")
            return None

In [47]:
import re
def _extract_json_response(response_text: str) -> Dict[str, Any]:
    """
    Extract structured JSON from LLM response.

    Args:
        response_text: Raw LLM response.

    Returns:
        Parsed JSON dict, or empty dict if parsing fails.
    """
    response_text = re.sub(r'<think>.*?</think>', '', response_text, flags=re.DOTALL).strip()
    # Try to find JSON block
    json_match = re.search(r"\{.*\}", response_text, re.DOTALL)
    if json_match:
        try:
            return json.loads(json_match.group())
        except json.JSONDecodeError:
            pass

    # Fallback: treat entire response as potential JSON
    try:
        return json.loads(response_text)
    except json.JSONDecodeError:
        return {}
    
def extract_code_from_response(response_text: str) -> str:
    """
    Extract Python code from response text.

    Looks for code wrapped in <code> and </code> tags,
    or in a JSON 'code' field.
    """
    # Try <code> tags first
    if "<code>" in response_text and "</code>" in response_text:
        start = response_text.find("<code>") + 6
        end = response_text.find("</code>")
        code = response_text[start:end].strip()
        return code

    # Try JSON format
    if '"code"' in response_text or "'code'" in response_text:
        import json
        import re

        # Find JSON-like structure
        json_match = re.search(r"\{.*\}", response_text, re.DOTALL)
        if json_match:
            try:
                json_obj = json.loads(json_match.group())
                if "code" in json_obj:
                    return json_obj["code"]
            except json.JSONDecodeError:
                pass

    # Fallback: return as-is
    return response_text

In [36]:
from prompts import SYSTEM_PROMPT, REASONING_PROMPT_TEMPLATE, REFINEMENT_PROMPT_TEMPLATE

user_message = REASONING_PROMPT_TEMPLATE.format(
            examples=examples_str, test_input=test_input_str
        )

In [35]:
len(user_message)

2359

In [39]:
response =  await _call_llm(SYSTEM_PROMPT, user_message)

USE LOCAL_LLM: True, MODEL: Qwen/Qwen3-14B


In [49]:
response_dict = _extract_json_response(response)
code = extract_code_from_response(
    response_dict.get("code", "") or response
)

In [51]:
# print(response)

In [50]:
response_dict

{'perception': 'The input and output grids show that numbers 1, 9, and 6 are mirrored across the vertical axis. For example, a 1 at (0,0) is mirrored to (0,7), a 9 at (1,1) is mirrored to (1,6), and a 6 at (5,1) is mirrored to (5,6). Other numbers (like 0) remain unchanged.',
 'hypothesis_1': 'For each cell containing 1, 9, or 6, mirror its value to the opposite side of the same row.',
 'hypothesis_2': 'Add mirrored copies of 1, 9, and 6 across the vertical axis, preserving their original positions.',
 'hypothesis_3': "The transformation involves reflecting specific values (1, 9, 6) symmetrically across the grid's vertical center.",
 'chosen_hypothesis': 'For each cell containing 1, 9, or 6, mirror its value to the opposite side of the same row.',
 'code': 'import numpy as np\n\ndef transform(grid):\n    for i in range(8):\n        for j in range(8):\n            if grid[i][j] in {1, 9, 6}:\n                mirrored_j = 7 - j\n                grid[i][mirrored_j] = grid[i][j]\n    retur

In [54]:
from sandbox import run_verification


success, error_msg = run_verification(code, train_pairs)

In [87]:
success, error_msg = out_run_verification(code, train_pairs)

In [88]:
success

True

In [90]:
from sandbox import run_test_inference


print("\n[STEP 5] Running on test input...")
test_input = test_cases[0]["input"]
result = run_test_inference(code, test_input)


[STEP 5] Running on test input...


In [91]:
result

(False,
 'Error during test inference:\nTraceback (most recent call last):\n  File "/home/abdul/ARC/arc_solver/sandbox.py", line 225, in run_test_inference\n    try:\n        ^\n  File "<string>", line 1, in <module>\nImportError: __import__ not found\n')

In [89]:
print(error_msg)

None


In [59]:
import numpy as np

def transform(grid):
    for i in range(8):
        for j in range(8):
            if grid[i][j] in {1, 9, 6}:
                mirrored_j = 7 - j
                grid[i][mirrored_j] = grid[i][j]
    return grid

In [65]:
transform(train_pairs[0]['input'])

[[1, 0, 0, 0, 0, 0, 0, 1],
 [0, 9, 0, 0, 0, 0, 9, 0],
 [0, 0, 1, 0, 0, 1, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0],
 [9, 6, 0, 0, 0, 0, 6, 9],
 [0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0]]

In [66]:
train_pairs[0]['output']

[[1, 0, 0, 0, 0, 0, 0, 1],
 [0, 9, 0, 0, 0, 0, 9, 0],
 [0, 0, 1, 0, 0, 1, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0],
 [9, 6, 0, 0, 0, 0, 6, 9],
 [0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0]]

In [85]:
# Create isolated namespace for execution
sandbox_namespace = {
    "np": np,
    "numpy": np,
    "__builtins__": {
        "__import__": __import__,
        "print": print,
        "len": len,
        "range": range,
        "int": int,
        "float": float,
        "list": list,
        "dict": dict,
        "tuple": tuple,
        "str": str,
        "bool": bool,
    },
}
exec(code, sandbox_namespace)


# try:
#     # Execute the code to define the transform function
#     exec(code, sandbox_namespace)
# except SyntaxError as e:
#     error_msg = f"SyntaxError in code:\n{e.msg}\nLine {e.lineno}: {e.text}"
#     return False, error_msg
# except Exception as e:
#     error_msg = f"Error during code definition:\n{traceback.format_exc()}"
#     return False, error_msg

In [86]:

import traceback
from typing import Tuple, Union

from httpx import TimeoutException

def out_run_verification(code: str, train_pairs: list) -> Union[Tuple[bool, None], Tuple[bool, str]]:
    """
    Execute generated code and verify against training pairs.

    Args:
        code: Python code string containing a `transform(grid)` function.
        train_pairs: List of {'input': grid, 'output': grid} dicts.

    Returns:
        Tuple of (success: bool, error_message: str or None)
        - If success: (True, None)
        - If failure: (False, detailed_error_message)
    """
    if not code:
        return False, "No code provided for verification."

    if len(code) > config.MAX_CODE_LENGTH:
        return (
            False,
            f"Code exceeds maximum length of {config.MAX_CODE_LENGTH} characters.",
        )

    # Create isolated namespace for execution
    sandbox_namespace = {
        "np": np,
        "numpy": np,
        "__builtins__": {
            "__import__": __import__,
            "print": print,
            "len": len,
            "range": range,
            "int": int,
            "float": float,
            "list": list,
            "dict": dict,
            "tuple": tuple,
            "str": str,
            "bool": bool,
        },
    }

    try:
        # Execute the code to define the transform function
        exec(code, sandbox_namespace)
    except SyntaxError as e:
        error_msg = f"SyntaxError in code:\n{e.msg}\nLine {e.lineno}: {e.text}"
        return False, error_msg
    except Exception as e:
        error_msg = f"Error during code definition:\n{traceback.format_exc()}"
        return False, error_msg

    # Check if transform function exists
    if "transform" not in sandbox_namespace:
        return False, "No 'transform' function defined in the provided code."

    transform_fn = sandbox_namespace["transform"]

    # Verify against each training pair
    for idx, pair in enumerate(train_pairs):
        try:
            train_input = normalize_grid(pair["input"])
            expected_output = normalize_grid(pair["output"])

            # Call the transform function with timeout protection
            try:
                actual_output = transform_fn(train_input)
            except TimeoutException:
                return (
                    False,
                    f"Code execution timeout on training example {idx + 1}",
                )

            # Normalize the output
            if actual_output is None:
                return (
                    False,
                    f"transform() returned None on training example {idx + 1}",
                )

            actual_output = normalize_grid(actual_output)

            # Check if shapes match
            if actual_output.shape != expected_output.shape:
                return (
                    False,
                    f"Shape mismatch on training example {idx + 1}:\n"
                    f"  Input shape: {train_input.shape}\n"
                    f"  Expected output shape: {expected_output.shape}\n"
                    f"  Got: {actual_output.shape}",
                )

            # Check if values match
            if not grids_equal(actual_output, expected_output):
                diff_mask = actual_output != expected_output
                num_diffs = np.sum(diff_mask)
                return (
                    False,
                    f"Output mismatch on training example {idx + 1}:\n"
                    f"  {num_diffs} cells differ\n"
                    f"  Expected:\n{expected_output}\n"
                    f"  Got:\n{actual_output}",
                )

        except Exception as e:
            error_msg = (
                f"Error executing transform on training example {idx + 1}:\n"
                f"{traceback.format_exc()}"
            )
            return False, error_msg

    # All verifications passed
    return True, None

In [75]:
def normalize_grid(grid) -> np.ndarray:
    """Normalize grid to numpy array with integer dtype."""
    arr = np.array(grid, dtype=np.int32)
    return arr

def grids_equal(grid1, grid2) -> bool:
    """Check if two grids are equal."""
    try:
        arr1 = normalize_grid(grid1)
        arr2 = normalize_grid(grid2)
        return np.array_equal(arr1, arr2)
    except Exception:
        return False

In [69]:
for idx, pair in enumerate(train_pairs):
    try:
        train_input = normalize_grid(pair["input"])
        expected_output = normalize_grid(pair["output"])
        print(f"Training example {idx + 1} input shape: {train_input.shape}, output shape: {expected_output.shape}")
    except Exception as e:
        error_msg = (
            f"Error executing transform on training example {idx + 1}:\n"
            f"{traceback.format_exc()}"
        )

Training example 1 input shape: (8, 8), output shape: (8, 8)
Training example 2 input shape: (8, 8), output shape: (8, 8)
Training example 3 input shape: (8, 8), output shape: (8, 8)


In [70]:
actual_output = transform(train_input)

In [74]:
# Normalize the output
if actual_output is None:
    print("transform() returned None on training example")

actual_output = normalize_grid(actual_output)

# Check if shapes match
if actual_output.shape != expected_output.shape:
    print(f"Shape mismatch on training example {idx + 1},\nInput shape: {train_input.shape}\n Expected output shape: {expected_output.shape}\n Got: {actual_output.shape}")
else:
    print("Shape matches expected output.")

Shape matches expected output.


In [76]:
if not grids_equal(actual_output, expected_output):
    print('Grids not equal')
else:
    print("got expected")

got expected
